In [ ]:
if input("Run? [N/y]\n") != 'y':
  # trow exception
  raise Exception("User aborted")

#########################################################################################################
# ++++++++++++++++++++++++++++++ CSS WORD WRAP FOR READABILITY +++++++++++++++++++++++++++++++++++++++++#
#########################################################################################################
from IPython.display import HTML, display
!pip install llama-cpp-python
def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)
#########################################################################################################
#########################################################################################################

import requests
import os

!rm -rf llama.cpp
# Clone the repository
!git clone https://github.com/ggml-org/llama.cpp

# Change directory to llama.cpp
%cd llama.cpp

!cmake -B build -DGGML_CUDA=ON
!cmake --build build --config Release -- -j$(nproc)

!pip install huggingface_hub hf_transfer
%env HF_HUB_ENABLE_HF_TRANSFER=1
#!huggingface-cli download bartowski/mistral-community_pixtral-12b-GGUF --include "mistral-community_pixtral-12b-Q6_K.gguf" --local-dir . --local-dir-use-symlinks False
#!huggingface-cli download ggml-org/pixtral-12b-GGUF --include "mmproj-pixtral-12b-f16.gguf" --local-dir . --local-dir-use-symlinks False
#!huggingface-cli download ggml-org/pixtral-12b-GGUF --include "pixtral-12b-f16.gguf" --local-dir . --local-dir-use-symlinks False
!huggingface-cli download bartowski/mistralai_Mistral-Small-3.1-24B-Instruct-2503-GGUF --include "mistralai_Mistral-Small-3.1-24B-Instruct-2503-Q6_K_L.gguf" --local-dir . --local-dir-use-symlinks False
!huggingface-cli download bartowski/mistralai_Mistral-Small-3.1-24B-Instruct-2503-GGUF --include "mmproj-mistralai_Mistral-Small-3.1-24B-Instruct-2503-f16.gguf" --local-dir . --local-dir-use-symlinks False
#!wget https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz
!wget https://antidote.cloud/f/d5d4d7c6e17d41198522/?dl=1 -O CrisisMMD_v2.0.tar.gz
!tar -xf CrisisMMD_v2.0.tar.gz

%cd /content/llama.cpp


Run? [N/y]
y
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 MB 36.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.9-cp311-cp311-linux_x86_64.whl size=4123157 sha256=e0b573ec128c844e6c33b7a7128a61ba7a354b3d8df55093638a8e5b119f4b01
  Stored in directory: /root/.cache/pip/wheels/9e/8f/bf/148c8eb7d69021eccd6eae6444f3accd48347587054ffd24e5
Successfully built llama-cpp-python
Cloning into 'llama.cpp'...
remote: Enumerating objects: 51306, done.
remote: Counting objects: 100% (223/223), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 51306 (delta 133), reused 50 (delta 43), pack-reused 51083 (from 4)
Receiving objects: 100% (51306/51306), 117.02 MiB | 37.83 MiB/s, done.
R

In [ ]:
# First, find and kill the current server
!pkill -f llama-server

# Wait a moment for it to fully terminate
!sleep 3

!nohup /content/llama.cpp/build/bin/llama-server --chat-template mistral-v7-tekken\
  -m /content/llama.cpp/mistralai_Mistral-Small-3.1-24B-Instruct-2503-Q6_K_L.gguf \
  --mmproj /content/llama.cpp/mmproj-mistralai_Mistral-Small-3.1-24B-Instruct-2503-f16.gguf \
  --host 0.0.0.0 \
  --port 8000 \
  -ngl 99 \
  -c 24576 \
  -b 1024 \
  -ub 512 \
  -cb \
  -t 8 \
  --mlock \
  --parallel 16 \
  > server.log 2>&1 &

# Check if it's running
!sleep 20
!ps aux | grep llama-server


root        8965 35.4  1.5 66470844 1344812 ?    Sl   14:03   0:07 /content/llama.cpp/build/bin/llama-server --chat-template mistral-v7-tekken -m /content/llama.cpp/mistralai_Mistral-Small-3.1-24B-Instruct-2503-Q6_K_L.gguf --mmproj /content/llama.cpp/mmproj-mistralai_Mistral-Small-3.1-24B-Instruct-2503-f16.gguf --host 0.0.0.0 --port 8000 -ngl 99 -c 24576 -b 1024 -ub 512 -cb -t 8 --mlock --parallel 16
root        9073  0.0  0.0   7376  3532 ?        S    14:04   0:00 /bin/bash -c ps aux | grep llama-server
root        9075  0.0  0.0   6484  2416 ?        S    14:04   0:00 grep llama-server


In [ ]:
import pandas as pd
import requests
import base64
import json
import os
import io
import re
import time
import random
import traceback
from PIL import Image
from tqdm.notebook import tqdm  # For Colab
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
import concurrent.futures

from google.colab import files
import os
import time
import threading
import zipfile
import glob

# Function to create a zip backup of results (without automatic download)
def create_backup(output_dir, backup_name=None):
    """Create a zip file of results without downloading it"""
    if backup_name is None:
        # Create a timestamp for the backup
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        backup_name = f"disaster_results_{timestamp}.zip"

    # Path for the zip file
    zip_path = f"/content/{backup_name}"

    print(f"\nCreating backup: {backup_name}...")

    # Create the zip file
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Walk through the results directory
        for root, _, files_list in os.walk(output_dir):
            for file in files_list:
                # Get the full path of the file
                file_path = os.path.join(root, file)
                # Calculate path inside the zip file
                arcname = os.path.relpath(file_path, os.path.dirname(output_dir))
                # Add file to the zip
                zipf.write(file_path, arcname)

    print(f"Backup created: {zip_path}")
    print(f"To download, run: files.download('{zip_path}')")

    return zip_path

# Function to run periodic backups in the background
def start_periodic_backups(output_dir, interval_minutes=15):
    """Start a background thread that creates periodic backups"""
    def backup_thread():
        while True:
            # Sleep for the specified interval
            time.sleep(interval_minutes * 60)

            # Create backup
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            backup_name = f"disaster_results_auto_{timestamp}.zip"

            try:
                backup_path = create_backup(output_dir, backup_name)
                print(f"Automatic backup completed at {time.strftime('%H:%M:%S')}")
                print(f"To download this backup, run: files.download('{backup_path}')")
            except Exception as e:
                print(f"Automatic backup failed: {e}")

    # Create and start the thread
    backup_thread = threading.Thread(target=backup_thread, daemon=True)
    backup_thread.start()

    print(f"Automatic backups will run every {interval_minutes} minutes")
    return backup_thread

# Function to list and download all results files individually
def download_all_results():
    """List and download all CSV, JSON, and PNG files in the results directory"""
    result_files = glob.glob("/content/llama.cpp/results/*.csv") + \
                  glob.glob("/content/llama.cpp/results/*.json") + \
                  glob.glob("/content/llama.cpp/results/*.png")

    print(f"Found {len(result_files)} result files")

    if not result_files:
        print("No result files found to download")
        return

    # Create a markdown cell listing all files
    files_markdown = "## Result Files\n\n"
    files_markdown += "Run any of these commands to download a specific file:\n\n"

    for file in result_files:
        filename = os.path.basename(file)
        files_markdown += f"```python\nfiles.download('{file}')  # Download {filename}\n```\n\n"

    # Display the markdown
    from IPython.display import Markdown, display
    display(Markdown(files_markdown))

    # Create a single zip with all results
    zip_path = create_backup("/content/llama.cpp/results", "all_results.zip")
    print(f"\nAll results are also packaged in: {zip_path}")
    print(f"To download the complete zip: files.download('{zip_path}')")

def find_first_image():
    """Helper function to find the correct path to the first image"""
    # Known paths to try
    possible_paths = [
        "/content/llama.cpp/data_image/california_wildfires/10_10_2017/917791044158185473_0.jpg",
        "/content/llama.cpp/CrisisMMD_v2.0/data_image/california_wildfires/10_10_2017/917791044158185473_0.jpg",
        "/content/CrisisMMD_v2.0/data_image/california_wildfires/10_10_2017/917791044158185473_0.jpg",
        "/content/data_image/california_wildfires/10_10_2017/917791044158185473_0.jpg"
    ]

    # Check each path
    for path in possible_paths:
        print(f"Checking {path}")
        if os.path.exists(path):
            print(f"Image found at: {path}")
            return path

    # Try to find by searching
    print("Searching for the image...")
    search_dirs = [
        "/content",
        "/content/llama.cpp",
        "/content/drive/MyDrive"
    ]

    for search_dir in search_dirs:
        if not os.path.exists(search_dir):
            continue

        for root, dirs, files in os.walk(search_dir):
            if "917791044158185473_0.jpg" in files:
                path = os.path.join(root, "917791044158185473_0.jpg")
                print(f"Found image at: {path}")
                return path

    print("Could not find the image automatically")
    return None

def clean_up_tsv_directory(tsv_dir):
    """Check for and optionally remove problematic files in the TSV directory"""
    print(f"Checking files in {tsv_dir}...")

    # List all files
    all_files = os.listdir(tsv_dir)
    print(f"Found {len(all_files)} total files")

    # Group files
    valid_tsv_files = []
    hidden_files = []
    non_tsv_files = []

    for filename in all_files:
        if filename.startswith('._'):
            hidden_files.append(filename)
        elif filename.endswith('.tsv'):
            valid_tsv_files.append(filename)
        else:
            non_tsv_files.append(filename)

    # Print summary
    print(f"Valid TSV files: {len(valid_tsv_files)}")
    for f in valid_tsv_files:
        print(f"  - {f}")

    print(f"Hidden files: {len(hidden_files)}")
    for f in hidden_files:
        print(f"  - {f}")

    print(f"Non-TSV files: {len(non_tsv_files)}")
    for f in non_tsv_files:
        print(f"  - {f}")

    # Ask to remove problematic files
    if hidden_files:
        remove_hidden = input("Do you want to remove hidden files? (y/n): ")
        if remove_hidden.lower() == 'y':
            for f in hidden_files:
                path = os.path.join(tsv_dir, f)
                try:
                    os.remove(path)
                    print(f"Removed: {f}")
                except Exception as e:
                    print(f"Error removing {f}: {e}")

    return valid_tsv_files

# Define the classifier function
def classify_with_schema(tweet_text, image_path, server_endpoint="http://localhost:8000/chat/completions"):
    """
    Classify using the OpenAI-compatible /chat/completions endpoint with a JSON schema
    """
    try:
        # Load and resize image
        image = Image.open(image_path)

        # Convert RGBA to RGB if needed
        if image.mode == 'RGBA':
            background = Image.new('RGB', image.size, (255, 255, 255))
            background.paste(image, mask=image.split()[3])
            image = background
        elif image.mode != 'RGB':
            image = image.convert('RGB')

        image = image.resize((512, 512))

        # Convert to base64
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=85)
        image_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')

        # Create the payload with JSON schema
        request_data = {
            "model": "mistral-v7-tekken",
            "messages": [
                {
                    "role": "system",
                    "content": """You are an expert humanitarian disaster analyst with extensive experience in classifying disaster-related content. Your task is to accurately classify tweets and images based on objective evidence rather than emotional responses.

When analyzing images:
1. First, carefully examine the entire image for ALL visual evidence of disaster impacts
2. Look for people, damage, rescue activities, and other humanitarian elements BEFORE considering the "not_humanitarian" category
3. Pay particular attention to subtle signs of affected individuals, even if they are not prominently displayed
4. Only classify as "not_humanitarian" when you are certain there is no disaster-related content

Remember: Images showing ANY signs of disaster impact, even if subtle or in the background, should be classified as "informative" and given an appropriate category other than "not_humanitarian"."""
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": f"""Tweet: "{tweet_text}"

CLASSIFICATION CRITERIA:

STEP 1: Analyze the TWEET TEXT first:
--------------------------------------
1. Tweet informativeness - determine if the tweet contains SPECIFIC information:
   - informative: Contains SPECIFIC disaster impact/response evidence, facts, or details
   - not_informative: Generic statements, emotions only, unrelated content, no specific details

2. Tweet category - identify the DOMINANT content (choose only ONE):
   - affected_individuals: Mentions displaced people, survivors, emotional responses (NOT injured/dead)
   - infrastructure_and_utility_damage: References damaged buildings, roads, bridges, utilities
   - injured_or_dead_people: Reports injuries, deaths, or specific casualty numbers
   - missing_or_found_people: Mentions people who are missing, found, or rescued by name or count
   - not_humanitarian: Irrelevant content, ads, jokes, political messages, misinformation
   - other_relevant_information: Weather data, satellite images, locations without people/damage
   - rescue_volunteering_or_donation_effort: Mentions donations, rescue missions, aid, volunteers
   - vehicle_damage: References damaged cars, trucks, ambulances, buses

STEP 2: Analyze the IMAGE separately and thoroughly:
---------------------------------------------------
1. Image informativeness - evaluate visual evidence:
   - informative: Contains ANY visual evidence of disaster impact/response (even if subtle)
   - not_informative: NO visual evidence of disaster impact (ONLY use if completely unrelated)

2. Image category - identify the DOMINANT visual element (choose only ONE):
   - affected_individuals: People appearing distressed/displaced WITHOUT visible injuries
   - infrastructure_and_utility_damage: Visible damage to buildings, roads, bridges
   - injured_or_dead_people: Visible injuries, bodies, medical care
   - missing_or_found_people: Search/rescue scenes, missing posters
   - not_humanitarian: ONLY use when image shows NO disaster-related content whatsoever
   - other_relevant_information: Maps, satellite images, generic disaster scenes without people/objects
   - rescue_volunteering_or_donation_effort: Aid distribution, rescue operations, emergency responders
   - vehicle_damage: Visibly damaged vehicles of any type

3. Damage severity (objective visual assessment):
   - little_or_no_damage: NO VISIBLE structural damage in the image
   - mild_damage: MINOR visible damage (cracking, partial damage, leaning)
   - severe_damage: MAJOR structural collapse or complete destruction
   - dont_know_or_cant_judge: ONLY use when damage cannot be assessed (no structures visible)

IMPORTANT REMINDERS:
- The tweet and image should be analyzed INDEPENDENTLY - they may belong to different categories
- Only classify as "not_humanitarian" when you are CERTAIN there is no disaster-related content
- Look carefully for subtle visual evidence before determining informativeness
- For damage severity, only consider VISIBLE evidence in the image itself

Return classification according to the specified JSON schema."""
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{image_base64}"
                            }
                        }
                    ]
                }
            ],
            "max_tokens": 4096,
            "temperature": 0,
            "response_format": {
                "type": "json_object",
                "schema": {
                    "type": "object",
                    "properties": {
                        "text_analysis": {
                            "type": "object",
                            "properties": {
                                "informativeness": {
                                    "type": "string",
                                    "enum": ["informative", "not_informative"]
                                },
                                "category": {
                                    "type": "string",
                                    "enum": [
                                        "affected_individuals",
                                        "infrastructure_and_utility_damage",
                                        "injured_or_dead_people",
                                        "missing_or_found_people",
                                        "not_humanitarian",
                                        "other_relevant_information",
                                        "rescue_volunteering_or_donation_effort",
                                        "vehicle_damage"
                                    ]
                                }
                            },
                            "required": ["informativeness", "category"]
                        },
                        "image_analysis": {
                            "type": "object",
                            "properties": {
                                "informativeness": {
                                    "type": "string",
                                    "enum": ["informative", "not_informative"]
                                },
                                "category": {
                                    "type": "string",
                                    "enum": [
                                        "affected_individuals",
                                        "infrastructure_and_utility_damage",
                                        "injured_or_dead_people",
                                        "missing_or_found_people",
                                        "not_humanitarian",
                                        "other_relevant_information",
                                        "rescue_volunteering_or_donation_effort",
                                        "vehicle_damage"
                                    ]
                                },
                                "damage_severity": {
                                    "type": "string",
                                    "enum": ["little_or_no_damage", "mild_damage", "severe_damage", "dont_know_or_cant_judge"]
                                }
                            },
                            "required": ["informativeness", "category", "damage_severity"]
                        }
                    },
                    "required": ["text_analysis", "image_analysis"]
                }
            }
        }

        # Send request to server with simplified retry logic
        for attempt in range(3):
            try:
                print(f"Sending request to {server_endpoint} (attempt {attempt+1}/3)")
                response = requests.post(server_endpoint, json=request_data, timeout=90)

                if response.status_code == 200:
                    result = response.json()
                    print(f"Response received. Status code: {response.status_code}")

                    # Extract content based on response structure
                    if "choices" in result and len(result["choices"]) > 0:
                        # Standard OpenAI API format
                        content = result["choices"][0]["message"]["content"]

                        try:
                            # Try parsing as JSON directly
                            parsed_result = json.loads(content)
                            return parsed_result
                        except json.JSONDecodeError:
                            # Try to extract JSON if it's wrapped in markdown
                            if "```json" in content and "```" in content:
                                json_match = re.search(r'```json\s*([\s\S]*?)\s*```', content)
                                if json_match:
                                    extracted_json = json_match.group(1).strip()
                                    return json.loads(extracted_json)

                    # Direct format where result itself is the expected JSON
                    if isinstance(result, dict) and "text_analysis" in result and "image_analysis" in result:
                        return result

                    # If we get here, response format is unknown
                    print(f"Unexpected response format. Response: {str(result)[:200]}...")
                    if attempt < 2:
                        wait_time = (2 ** attempt) + random.uniform(0, 1)
                        print(f"Retrying in {wait_time:.1f}s...")
                        time.sleep(wait_time)
                        continue
                    return None

                else:
                    print(f"Server error: {response.status_code} - {response.text[:100]}...")
                    if attempt < 2:
                        wait_time = (2 ** attempt) + random.uniform(0, 1)
                        print(f"Retrying in {wait_time:.1f}s...")
                        time.sleep(wait_time)
                    else:
                        return None

            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                if attempt < 2:
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"Network error: {e}. Retrying in {wait_time:.1f}s...")
                    time.sleep(wait_time)
                else:
                    print(f"Network error after 3 attempts: {e}")
                    return None
            except Exception as e:
                print(f"Error during request: {e}")
                traceback.print_exc()
                return None

        # If we get here, all attempts failed
        return None
    except Exception as e:
        print(f"Error processing image or request: {e}")
        traceback.print_exc()
        return None

def process_all_tsv_files(tsv_dir, base_dir, output_dir, max_samples=None, batch_size=8, max_workers=4, backup_interval=15):
    """
    Process all TSV files with automatic backups
    """
    import concurrent.futures

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    # Start periodic backups
    if backup_interval > 0:
        start_periodic_backups(output_dir, interval_minutes=backup_interval)

    # Get all TSV files in the directory, filtering out hidden files
    tsv_files = [f for f in os.listdir(tsv_dir) if f.endswith('.tsv') and not f.startswith('._')]
    print(f"Found {len(tsv_files)} TSV files to process")

    # First, count total items across all TSV files to set up a global progress bar
    print("Counting total items across all TSV files...")
    total_items = 0
    tsv_item_counts = {}

    for tsv_file in tsv_files:
        tsv_path = os.path.join(tsv_dir, tsv_file)
        try:
            # Try UTF-8 first
            df = pd.read_csv(tsv_path, sep='\t', encoding='utf-8')
        except UnicodeDecodeError:
            try:
                # Try latin-1 as fallback
                df = pd.read_csv(tsv_path, sep='\t', encoding='latin-1')
            except Exception as e:
                print(f"Error loading {tsv_file}: {e}")
                continue

        # If max_samples is specified, adjust count accordingly
        file_count = min(len(df), max_samples) if max_samples else len(df)
        tsv_item_counts[tsv_file] = file_count
        total_items += file_count

    print(f"Total items to process: {total_items}")

    # Create a global progress bar
    global_pbar = tqdm(total=total_items, desc="Overall progress", position=0)

    all_results = []
    processed_items = 0

    for tsv_file in tsv_files:
        tsv_path = os.path.join(tsv_dir, tsv_file)
        disaster_name = tsv_file.replace('_final_data.tsv', '')

        print(f"\nProcessing {disaster_name}...")

        # Load TSV file with explicit encoding and error handling
        try:
            # Try UTF-8 first
            df = pd.read_csv(tsv_path, sep='\t', encoding='utf-8')
        except UnicodeDecodeError:
            try:
                # Try latin-1 as fallback
                print(f"UTF-8 decoding failed, trying latin-1 encoding for {tsv_file}")
                df = pd.read_csv(tsv_path, sep='\t', encoding='latin-1')
            except Exception as e:
                print(f"Error loading {tsv_file}: {e}")
                continue

        file_item_count = len(df)
        print(f"Loaded {file_item_count} rows from {tsv_file}")

        # Ensure all required columns are present
        required_columns = ['tweet_id', 'image_id', 'tweet_text', 'image_path',
                           'text_info', 'text_human', 'image_info', 'image_human']

        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            print(f"Warning: Missing required columns in {tsv_file}: {missing_columns}")
            print("Skipping this file")
            continue

        # Limit samples if needed
        if max_samples:
            df = df.sample(min(max_samples, len(df)), random_state=42)
            print(f"Sampled {len(df)} rows")

        # Create batches
        batches = [df.iloc[i:i+batch_size].to_dict('records') for i in range(0, len(df), batch_size)]

        disaster_results = []

        # Create file-specific progress bar
        file_pbar = tqdm(total=len(df), desc=f"Processing {disaster_name}", position=1, leave=False)

        # Process batches with ThreadPoolExecutor
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(process_batch, batch, base_dir): batch for batch in batches}

            for future in concurrent.futures.as_completed(futures):
                try:
                    batch_results = future.result()
                    batch = futures[future]
                    batch_size = len(batch)

                    if batch_results:
                        disaster_results.extend(batch_results)

                        # Update both progress bars
                        file_pbar.update(batch_size)
                        global_pbar.update(batch_size)

                        # Save intermediate results periodically
                        if len(disaster_results) % 50 == 0:
                            intermediate_df = pd.DataFrame(disaster_results)
                            intermediate_file = os.path.join(output_dir, f"{disaster_name}_intermediate.csv")
                            intermediate_df.to_csv(intermediate_file, index=False)
                            print(f"Saved {len(disaster_results)} intermediate results to {intermediate_file}")
                    else:
                        # If no results were returned, still update progress
                        file_pbar.update(batch_size)
                        global_pbar.update(batch_size)

                except Exception as e:
                    # If an error occurs, still update progress for this batch
                    batch = futures[future]
                    batch_size = len(batch)
                    file_pbar.update(batch_size)
                    global_pbar.update(batch_size)

                    print(f"Error processing batch: {e}")
                    traceback.print_exc()

        # Close file progress bar
        file_pbar.close()

        # Save disaster-specific results
        if disaster_results:
            disaster_df = pd.DataFrame(disaster_results)
            disaster_file = os.path.join(output_dir, f"{disaster_name}_results.csv")
            disaster_df.to_csv(disaster_file, index=False)
            print(f"Saved {len(disaster_results)} results to {disaster_file}")

            all_results.extend(disaster_results)
        else:
            print(f"No valid results for {disaster_name}")

    # Close global progress bar
    global_pbar.close()

    # Save combined results
    if all_results:
        all_df = pd.DataFrame(all_results)
        all_file = os.path.join(output_dir, "all_results.csv")
        all_df.to_csv(all_file, index=False)
        print(f"Saved {len(all_results)} total results to {all_file}")

    # At the end of processing, create a final backup
    print("\nProcessing complete! Creating final backup...")
    final_backup_path = create_backup(output_dir, "disaster_results_final.zip")
    print(f"Final backup created: {final_backup_path}")
    print(f"To download the complete results, run: files.download('{final_backup_path}')")

    return all_results

from sklearn.metrics import f1_score, classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def calculate_f1_scores(results_df):
    """Calculate F1 scores comparing model predictions to ground truth"""
    metrics = {}

    # Filter out rows with missing predictions or ground truth
    has_text_preds = pd.notna(results_df['predicted_text_info']) & pd.notna(results_df['predicted_text_human'])
    has_text_truth = pd.notna(results_df['true_text_info']) & pd.notna(results_df['true_text_human'])
    has_image_preds = pd.notna(results_df['predicted_image_info']) & pd.notna(results_df['predicted_image_human'])
    has_image_truth = pd.notna(results_df['true_image_info']) & pd.notna(results_df['true_image_human'])

    # Create filtered dataframes with only valid rows
    text_df = results_df[has_text_preds & has_text_truth]
    image_df = results_df[has_image_preds & has_image_truth]

    # Calculate text informativeness F1 (binary classification)
    if len(text_df) > 0:
        text_info_f1 = f1_score(
            text_df['true_text_info'],
            text_df['predicted_text_info'],
            pos_label='informative',
            average='binary'
        )
        metrics['text_info_f1'] = text_info_f1

        # Calculate text category F1 (multiclass classification)
        text_human_f1 = f1_score(
            text_df['true_text_human'],
            text_df['predicted_text_human'],
            average='weighted'  # Use weighted average for imbalanced classes
        )
        metrics['text_human_f1'] = text_human_f1

    # Calculate image informativeness F1 (binary classification)
    if len(image_df) > 0:
        image_info_f1 = f1_score(
            image_df['true_image_info'],
            image_df['predicted_image_info'],
            pos_label='informative',
            average='binary'
        )
        metrics['image_info_f1'] = image_info_f1

        # Calculate image category F1 (multiclass classification)
        image_human_f1 = f1_score(
            image_df['true_image_human'],
            image_df['predicted_image_human'],
            average='weighted'  # Use weighted average for imbalanced classes
        )
        metrics['image_human_f1'] = image_human_f1

    # Calculate damage severity F1 (only for rows with damage annotations)
    if 'true_image_damage' in results_df.columns:
        has_damage_truth = pd.notna(results_df['true_image_damage'])
        has_damage_pred = pd.notna(results_df['predicted_image_damage'])
        damage_df = results_df[has_damage_truth & has_damage_pred]

        if len(damage_df) > 0:
            damage_f1 = f1_score(
                damage_df['true_image_damage'],
                damage_df['predicted_image_damage'],
                average='weighted'  # Use weighted average for imbalanced classes
            )
            metrics['damage_f1'] = damage_f1

    return metrics

def evaluate_results(results_file):
    """Evaluate model performance against ground truth with detailed metrics"""
    if not os.path.exists(results_file):
        print(f"Results file not found: {results_file}")
        return

    df = pd.read_csv(results_file)
    print(f"Loaded {len(df)} results from {results_file}")

    # Basic statistics

    # Count by disaster type
    if 'disaster_type' in df.columns:
        disaster_counts = df['disaster_type'].value_counts()
        print("\nResults by disaster type:")
        for disaster, count in disaster_counts.items():
            print(f"  - {disaster}: {count}")

    # Count predictions by category
    print("\nText category predictions:")
    if 'predicted_text_human' in df.columns:
        text_counts = df['predicted_text_human'].value_counts()
        for category, count in text_counts.items():
            print(f"  - {category}: {count} ({count/len(df):.1%})")

    print("\nImage category predictions:")
    if 'predicted_image_human' in df.columns:
        image_counts = df['predicted_image_human'].value_counts()
        for category, count in image_counts.items():
            print(f"  - {category}: {count} ({count/len(df):.1%})")

    print("\nDamage severity predictions:")
    if 'predicted_image_damage' in df.columns:
        damage_counts = df['predicted_image_damage'].value_counts()
        for severity, count in damage_counts.items():
            print(f"  - {severity}: {count} ({count/len(df):.1%})")

    # Performance evaluation with F1 scores
    print("\n===== MODEL PERFORMANCE (F1 SCORES) =====")
    metrics = calculate_f1_scores(df)

    if 'text_info_f1' in metrics:
        print(f"Text Informativeness F1: {metrics['text_info_f1']:.4f}")
    if 'text_human_f1' in metrics:
        print(f"Text Category F1: {metrics['text_human_f1']:.4f}")
    if 'image_info_f1' in metrics:
        print(f"Image Informativeness F1: {metrics['image_info_f1']:.4f}")
    if 'image_human_f1' in metrics:
        print(f"Image Category F1: {metrics['image_human_f1']:.4f}")
    if 'damage_f1' in metrics:
        print(f"Damage Severity F1: {metrics['damage_f1']:.4f}")

    # Save metrics to file
    metrics_file = results_file.replace('.csv', '_metrics.json')
    with open(metrics_file, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"Metrics saved to {metrics_file}")

    # Generate detailed classification reports
    print("\n===== DETAILED CLASSIFICATION REPORTS =====")

    # Filter valid rows
    has_text_preds = pd.notna(df['predicted_text_info']) & pd.notna(df['predicted_text_human'])
    has_text_truth = pd.notna(df['true_text_info']) & pd.notna(df['true_text_human'])
    has_image_preds = pd.notna(df['predicted_image_info']) & pd.notna(df['predicted_image_human'])
    has_image_truth = pd.notna(df['true_image_info']) & pd.notna(df['true_image_human'])

    text_df = df[has_text_preds & has_text_truth]
    image_df = df[has_image_preds & has_image_truth]

    if len(text_df) > 0:
        print("\nText Informativeness Classification Report:")
        print(classification_report(
            text_df['true_text_info'],
            text_df['predicted_text_info']
        ))

        print("\nText Category Classification Report:")
        print(classification_report(
            text_df['true_text_human'],
            text_df['predicted_text_human']
        ))

    if len(image_df) > 0:
        print("\nImage Informativeness Classification Report:")
        print(classification_report(
            image_df['true_image_info'],
            image_df['predicted_image_info']
        ))

        print("\nImage Category Classification Report:")
        print(classification_report(
            image_df['true_image_human'],
            image_df['predicted_image_human']
        ))

    if 'true_image_damage' in df.columns:
        has_damage_truth = pd.notna(df['true_image_damage'])
        has_damage_pred = pd.notna(df['predicted_image_damage'])
        damage_df = df[has_damage_truth & has_damage_pred]

        if len(damage_df) > 0:
            print("\nDamage Severity Classification Report:")
            print(classification_report(
                damage_df['true_image_damage'],
                damage_df['predicted_image_damage']
            ))

    # Create and save confusion matrices
    try:
        generate_confusion_matrices(df, results_file.replace('.csv', '_confusion_matrices.png'))
        print(f"Confusion matrices saved to {results_file.replace('.csv', '_confusion_matrices.png')}")
    except Exception as e:
        print(f"Error generating confusion matrices: {e}")

    return metrics

def generate_confusion_matrices(df, output_file):
    """Generate and save confusion matrices for model predictions"""
    # Make sure matplotlib is available
    import matplotlib.pyplot as plt
    import seaborn as sns

    # Create a figure with multiple subplots
    plt.figure(figsize=(20, 15))

    # Filter valid rows
    has_text_preds = pd.notna(df['predicted_text_info']) & pd.notna(df['predicted_text_human'])
    has_text_truth = pd.notna(df['true_text_info']) & pd.notna(df['true_text_human'])
    has_image_preds = pd.notna(df['predicted_image_info']) & pd.notna(df['predicted_image_human'])
    has_image_truth = pd.notna(df['true_image_info']) & pd.notna(df['true_image_human'])

    text_df = df[has_text_preds & has_text_truth]
    image_df = df[has_image_preds & has_image_truth]

    # Plot Text Informativeness Confusion Matrix
    if len(text_df) > 0:
        plt.subplot(2, 3, 1)
        conf_mat = confusion_matrix(
            text_df['true_text_info'],
            text_df['predicted_text_info'],
            normalize='true'  # Normalize by row (ground truth)
        )
        sns.heatmap(conf_mat, annot=True, fmt='.2f', cmap='Blues',
                   xticklabels=['not_informative', 'informative'],
                   yticklabels=['not_informative', 'informative'])
        plt.title('Text Informativeness')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')

    # Plot Image Informativeness Confusion Matrix
    if len(image_df) > 0:
        plt.subplot(2, 3, 2)
        conf_mat = confusion_matrix(
            image_df['true_image_info'],
            image_df['predicted_image_info'],
            normalize='true'  # Normalize by row (ground truth)
        )
        sns.heatmap(conf_mat, annot=True, fmt='.2f', cmap='Blues',
                   xticklabels=['not_informative', 'informative'],
                   yticklabels=['not_informative', 'informative'])
        plt.title('Image Informativeness')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')

    # Plot Text Category Confusion Matrix (top categories)
    if len(text_df) > 0:
        plt.subplot(2, 3, 3)
        # Get the most common categories (top 5)
        top_cats = pd.concat([text_df['true_text_human'], text_df['predicted_text_human']]).value_counts().head(5).index
        # Filter to only include top categories
        filtered_text_df = text_df[text_df['true_text_human'].isin(top_cats) & text_df['predicted_text_human'].isin(top_cats)]

        if len(filtered_text_df) > 0:
            conf_mat = confusion_matrix(
                filtered_text_df['true_text_human'],
                filtered_text_df['predicted_text_human'],
                normalize='true',  # Normalize by row (ground truth)
                labels=top_cats
            )
            sns.heatmap(conf_mat, annot=True, fmt='.2f', cmap='Blues',
                      xticklabels=top_cats, yticklabels=top_cats)
            plt.title('Text Category (Top 5)')
            plt.ylabel('True Label')
            plt.xlabel('Predicted Label')
            plt.xticks(rotation=45)
            plt.yticks(rotation=45)

    # Plot Image Category Confusion Matrix (top categories)
    if len(image_df) > 0:
        plt.subplot(2, 3, 4)
        # Get the most common categories (top 5)
        top_cats = pd.concat([image_df['true_image_human'], image_df['predicted_image_human']]).value_counts().head(5).index
        # Filter to only include top categories
        filtered_image_df = image_df[image_df['true_image_human'].isin(top_cats) & image_df['predicted_image_human'].isin(top_cats)]

        if len(filtered_image_df) > 0:
            conf_mat = confusion_matrix(
                filtered_image_df['true_image_human'],
                filtered_image_df['predicted_image_human'],
                normalize='true',  # Normalize by row (ground truth)
                labels=top_cats
            )
            sns.heatmap(conf_mat, annot=True, fmt='.2f', cmap='Blues',
                      xticklabels=top_cats, yticklabels=top_cats)
            plt.title('Image Category (Top 5)')
            plt.ylabel('True Label')
            plt.xlabel('Predicted Label')
            plt.xticks(rotation=45)
            plt.yticks(rotation=45)

    # Plot Damage Severity Confusion Matrix
    if 'true_image_damage' in df.columns:
        has_damage_truth = pd.notna(df['true_image_damage'])
        has_damage_pred = pd.notna(df['predicted_image_damage'])
        damage_df = df[has_damage_truth & has_damage_pred]

        if len(damage_df) > 0:
            plt.subplot(2, 3, 5)
            # Map model damage levels to ground truth format if needed
            model_map = {
                'little_or_no_damage': 'little_or_none',
                'mild_damage': 'mild',
                'severe_damage': 'severe',
                'dont_know_or_cant_judge': 'dont_know'
            }

            # Check if mapping is needed
            unique_pred = damage_df['predicted_image_damage'].unique()
            unique_true = damage_df['true_image_damage'].unique()

            # Apply mapping if needed
            if any(x in model_map for x in unique_pred) and not any(x in model_map for x in unique_true):
                damage_df = damage_df.copy()
                damage_df['predicted_image_damage'] = damage_df['predicted_image_damage'].map(lambda x: model_map.get(x, x))

            # Get common damage labels
            damage_labels = sorted(list(set(damage_df['true_image_damage'].unique()) | set(damage_df['predicted_image_damage'].unique())))

            conf_mat = confusion_matrix(
                damage_df['true_image_damage'],
                damage_df['predicted_image_damage'],
                normalize='true',  # Normalize by row (ground truth)
                labels=damage_labels
            )
            sns.heatmap(conf_mat, annot=True, fmt='.2f', cmap='Blues',
                      xticklabels=damage_labels, yticklabels=damage_labels)
            plt.title('Damage Severity')
            plt.ylabel('True Label')
            plt.xlabel('Predicted Label')
            plt.xticks(rotation=45)
            plt.yticks(rotation=45)

    # Add overall title
    plt.suptitle('Confusion Matrices - Model vs. Ground Truth', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to make room for the suptitle

    # Save to file
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.close()

def evaluate_by_disaster_type(results_file):
    """Calculate and compare F1 scores across different disaster types"""
    if not os.path.exists(results_file):
        print(f"Results file not found: {results_file}")
        return

    df = pd.read_csv(results_file)

    if 'disaster_type' not in df.columns:
        print("Disaster type information not available in results")
        return

    # Get unique disaster types
    disaster_types = df['disaster_type'].unique()

    print(f"\n===== PERFORMANCE BY DISASTER TYPE ({len(disaster_types)} types) =====")

    # Initialize results dictionary
    disaster_metrics = {}

    for disaster in disaster_types:
        # Filter data for this disaster type
        disaster_df = df[df['disaster_type'] == disaster]

        print(f"\n--- {disaster} ({len(disaster_df)} samples) ---")

        # Calculate metrics for this disaster type
        metrics = calculate_f1_scores(disaster_df)
        disaster_metrics[disaster] = metrics

        # Print metrics
        if 'text_info_f1' in metrics:
            print(f"Text Informativeness F1: {metrics['text_info_f1']:.4f}")
        if 'text_human_f1' in metrics:
            print(f"Text Category F1: {metrics['text_human_f1']:.4f}")
        if 'image_info_f1' in metrics:
            print(f"Image Informativeness F1: {metrics['image_info_f1']:.4f}")
        if 'image_human_f1' in metrics:
            print(f"Image Category F1: {metrics['image_human_f1']:.4f}")
        if 'damage_f1' in metrics:
            print(f"Damage Severity F1: {metrics['damage_f1']:.4f}")

    # Save metrics by disaster type
    metrics_by_disaster_file = results_file.replace('.csv', '_metrics_by_disaster.json')
    with open(metrics_by_disaster_file, 'w') as f:
        json.dump(disaster_metrics, f, indent=2)
    print(f"\nMetrics by disaster type saved to {metrics_by_disaster_file}")

    # Create a comparison dataframe for visualization
    try:
        # Convert to DataFrame for easier comparison
        metrics_df = pd.DataFrame.from_dict(disaster_metrics, orient='index')

        # Save as CSV
        metrics_csv = results_file.replace('.csv', '_metrics_by_disaster.csv')
        metrics_df.to_csv(metrics_csv)
        print(f"Metrics comparison CSV saved to {metrics_csv}")

        # Create a heatmap visualization
        plt.figure(figsize=(12, 8))
        sns.heatmap(metrics_df, annot=True, fmt='.3f', cmap='viridis', linewidths=.5)
        plt.title('F1 Scores by Disaster Type')
        plt.tight_layout()

        # Save the heatmap
        heatmap_file = results_file.replace('.csv', '_disaster_comparison.png')
        plt.savefig(heatmap_file, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Comparison heatmap saved to {heatmap_file}")
    except Exception as e:
        print(f"Error creating comparison visualization: {e}")

    return disaster_metrics

# Test function to verify the pipeline with a single example
def test_single_example(tsv_path, base_dir):
    """Test the pipeline with a single example to verify it works"""
    # Load the TSV file
    df = pd.read_csv(tsv_path, sep='\t')

    # Get the first row
    first_row = df.iloc[0].to_dict()

    print(f"Testing with tweet: {first_row['tweet_text']}")

    # Build image path - add CrisisMMD_v2.0/ prefix if not already included
    image_path_in_tsv = first_row['image_path']
    if not image_path_in_tsv.startswith("CrisisMMD_v2.0/"):
        image_path = os.path.join(base_dir, "CrisisMMD_v2.0", image_path_in_tsv)
    else:
        image_path = os.path.join(base_dir, image_path_in_tsv)

    print(f"Image path: {image_path}")

    if not os.path.exists(image_path):
        print(f"Error: Image not found at {image_path}")
        print("Trying alternative paths...")

        # Try some alternative path formats
        alternatives = [
            os.path.join(base_dir, "CrisisMMD_v2.0", image_path_in_tsv),
            os.path.join(base_dir, image_path_in_tsv),
            os.path.join("/content", "CrisisMMD_v2.0", image_path_in_tsv),
            "/content/llama.cpp/CrisisMMD_v2.0/" + image_path_in_tsv
        ]

        for alt_path in alternatives:
            print(f"Checking: {alt_path}")
            if os.path.exists(alt_path):
                print(f"Found image at: {alt_path}")
                image_path = alt_path
                break
        else:
            print("Could not find image with any path format")
            return

    # Run classification
    result = classify_with_schema(first_row['tweet_text'], image_path)

    if result:
        print("Classification successful!")
        print(json.dumps(result, indent=2))

        # Compare with ground truth
        print("\nComparison with ground truth:")
        print(f"Text informativeness - Model: {result['text_analysis']['informativeness']}, Human: {first_row['text_info']}")
        print(f"Text category - Model: {result['text_analysis']['category']}, Human: {first_row['text_human']}")
        print(f"Image informativeness - Model: {result['image_analysis']['informativeness']}, Human: {first_row['image_info']}")
        print(f"Image category - Model: {result['image_analysis']['category']}, Human: {first_row['image_human']}")

        return result
    else:
        print("Classification failed")
        return None

# Function to process a batch of rows - updated path handling
def process_batch(batch, base_dir):
    results = []

    for row in batch:
        try:
            # Get tweet info
            tweet_id = str(row['tweet_id'])
            image_id = str(row['image_id'])
            tweet_text = row['tweet_text']

            # Build full image path - add CrisisMMD_v2.0/ prefix if not already included
            image_path_in_tsv = row['image_path']
            if not image_path_in_tsv.startswith("CrisisMMD_v2.0/"):
                image_path = os.path.join(base_dir, "CrisisMMD_v2.0", image_path_in_tsv)
            else:
                image_path = os.path.join(base_dir, image_path_in_tsv)

            # Check if image exists
            if not os.path.exists(image_path):
                print(f"Image not found: {image_path}")

                # Try alternative path
                alt_path = "/content/llama.cpp/CrisisMMD_v2.0/" + image_path_in_tsv
                if os.path.exists(alt_path):
                    print(f"Found at alternative path: {alt_path}")
                    image_path = alt_path
                else:
                    print(f"Also not found at: {alt_path}")
                    continue

            # Classify the tweet and image
            result = classify_with_schema(tweet_text, image_path)

            if result:
                # Store the classification with human annotations for comparison
                processed_result = {
                    "tweet_id": tweet_id,
                    "image_id": image_id,
                    "tweet_text": tweet_text,
                    "image_path": row['image_path'],
                    "disaster_type": os.path.basename(os.path.dirname(row['image_path'])),

                    # Model predictions
                    "predicted_text_info": result["text_analysis"]["informativeness"],
                    "predicted_text_human": result["text_analysis"]["category"],
                    "predicted_image_info": result["image_analysis"]["informativeness"],
                    "predicted_image_human": result["image_analysis"]["category"],
                    "predicted_image_damage": result["image_analysis"]["damage_severity"],

                    # Ground truth values
                    "true_text_info": row['text_info'],
                    "true_text_human": row['text_human'],
                    "true_image_info": row['image_info'],
                    "true_image_human": row['image_human'],
                }

                # Only add damage if it exists in the dataset
                if 'image_damage' in row and not pd.isna(row['image_damage']):
                    processed_result["true_image_damage"] = row['image_damage']

                results.append(processed_result)
                print(f"Successfully processed {tweet_id}_{image_id}")
            else:
                print(f"Classification failed for {tweet_id}_{image_id}")

        except Exception as e:
            print(f"Error processing row: {e}")
            traceback.print_exc()

    return results

if __name__ == "__main__":
    # Configure paths
    TSV_DIR = "/content/llama.cpp/CrisisMMD_v2.0/annotations/"
    BASE_DIR = "/content/llama.cpp/"
    OUTPUT_DIR = "/content/llama.cpp/results/"

    # Add backup helper functions
    def manual_backup():
        """Create a manual backup of all results"""
        backup_path = create_backup(OUTPUT_DIR)
        return backup_path

    def download_backup(path):
        """Download a specified backup file"""
        if os.path.exists(path):
            print(f"Downloading {path}...")
            files.download(path)
        else:
            print(f"File not found: {path}")

    print("\nHelpful commands:")
    print("- manual_backup(): Create a backup of current results")
    print("- download_backup(path): Download a specific backup file")
    print("- files.download(path): Download any file directly")

    # Clean up the TSV directory first
    print("Checking TSV directory for problematic files...")
    valid_files = clean_up_tsv_directory(TSV_DIR)

    # First, try to locate the test image
    print("Attempting to locate a test image...")
    found_path = find_first_image()

    if found_path:
        # Extract the base directory from the found path
        if "CrisisMMD_v2.0" in found_path:
            BASE_DIR = found_path.split("CrisisMMD_v2.0")[0]
        print(f"Using base directory: {BASE_DIR}")

    # Test with a single example first
    print("Testing with a single example:")
    valid_tsv = None
    for tsv in valid_files:
        if "california_wildfires" in tsv and not tsv.startswith('._'):
            valid_tsv = tsv
            break

    if not valid_tsv:
        valid_tsv = valid_files[0] if valid_files else None

    if valid_tsv:
        test_file = os.path.join(TSV_DIR, valid_tsv)
        test_result = test_single_example(test_file, BASE_DIR)

        if test_result:
            # If test is successful, ask to process all files
            proceed = input("\nTest successful! Process all TSV files? (y/n): ")

            if proceed.lower() == 'y':
                print("\nProcessing all TSV files...")
                # You can adjust these parameters based on your resources
                all_results = process_all_tsv_files(
                    TSV_DIR,
                    BASE_DIR,
                    OUTPUT_DIR,
                    max_samples=None,  # Set to None to process all samples
                    batch_size=16,    # Adjust based on memory requirements
                    max_workers=16    # Adjust based on CPU/GPU resources
                )
            else:
                print("Processing canceled.")
        else:
            print("Test failed. Please check your setup and try again.")
    else:
        print("No valid TSV files found. Please check your dataset directory.")

    # REPLACE THE EXISTING "OPTIONALLY EVALUATE" SECTION WITH THIS:
    # After the main processing is complete, always evaluate and download results
    print("\n===== EVALUATING RESULTS =====")
    results_file = os.path.join(OUTPUT_DIR, "all_results.csv")

    # Check if results file exists
    if os.path.exists(results_file):
        print(f"Found results file: {results_file}")

        # Evaluate the results automatically
        print("\nCalculating performance metrics...")
        metrics = evaluate_results(results_file)

        # Evaluate performance by disaster type
        print("\nAnalyzing performance by disaster type...")
        disaster_metrics = evaluate_by_disaster_type(results_file)

        # Create and download a zip with all result files
        print("\nCreating results package for download...")
        results_backup = create_backup(OUTPUT_DIR, "disaster_analysis_results.zip")

        # Try to automatically download the results
        try:
            print("\nStarting download of all results...")
            files.download(results_backup)
            print("Download initiated. If you don't see the download prompt, the file is available at:")
            print(f"  {results_backup}")
        except Exception as e:
            print(f"\nAutomatic download failed: {e}")
            print("To download results manually, run this command:")
            print(f"  files.download('{results_backup}')")

        # Also provide individual download commands for key files
        key_files = [
            results_file,
            results_file.replace('.csv', '_metrics.json'),
            results_file.replace('.csv', '_metrics_by_disaster.csv'),
            results_file.replace('.csv', '_confusion_matrices.png'),
            results_file.replace('.csv', '_disaster_comparison.png')
        ]

        print("\nKey result files:")
        for f in key_files:
            if os.path.exists(f):
                filename = os.path.basename(f)
                print(f"  - {filename}: files.download('{f}')")

    else:
        print(f"Results file not found: {results_file}")

        # Look for any results files
        csv_files = glob.glob(os.path.join(OUTPUT_DIR, "*.csv"))
        if csv_files:
            print("\nFound these result files:")
            for csv_file in csv_files:
                print(f"  - {os.path.basename(csv_file)}")

            # Ask which file to evaluate
            print("\nTo evaluate a specific file, run:")
            print("  evaluate_results('/path/to/your/file.csv')")
            print("  evaluate_by_disaster_type('/path/to/your/file.csv')")
        else:
            print("\nNo result files found in the output directory.")


Helpful commands:
- manual_backup(): Create a backup of current results
- download_backup(path): Download a specific backup file
- files.download(path): Download any file directly
Checking TSV directory for problematic files...
Checking files in /content/llama.cpp/CrisisMMD_v2.0/annotations/...
Found 11 total files
Valid TSV files: 7
  - mexico_earthquake_final_data.tsv
  - hurricane_maria_final_data.tsv
  - hurricane_irma_final_data.tsv
  - srilanka_floods_final_data.tsv
  - iraq_iran_earthquake_final_data.tsv
  - california_wildfires_final_data.tsv
  - hurricane_harvey_final_data.tsv
Hidden files: 4
  - ._california_wildfires_final_data.tsv
  - ._hurricane_irma_final_data.tsv
  - ._hurricane_maria_final_data.tsv
  - ._srilanka_floods_final_data.tsv
Non-TSV files: 0
Do you want to remove hidden files? (y/n): y
Removed: ._california_wildfires_final_data.tsv
Removed: ._hurricane_irma_final_data.tsv
Removed: ._hurricane_maria_final_data.tsv
Removed: ._srilanka_floods_final_data.tsv
Atte

Overall progress:   0%|          | 0/18082 [00:00<?, ?it/s]


Processing mexico_earthquake...
Loaded 1380 rows from mexico_earthquake_final_data.tsv


Processing mexico_earthquake:   0%|          | 0/1380 [00:00<?, ?it/s]

Sending request to http://localhost:8000/chat/completions (attempt 1/3)
Sending request to http://localhost:8000/chat/completions (attempt 1/3)
Sending request to http://localhost:8000/chat/completions (attempt 1/3)
Sending request to http://localhost:8000/chat/completions (attempt 1/3)
Sending request to http://localhost:8000/chat/completions (attempt 1/3)
Sending request to http://localhost:8000/chat/completions (attempt 1/3)
Sending request to http://localhost:8000/chat/completions (attempt 1/3)
Sending request to http://localhost:8000/chat/completions (attempt 1/3)
Sending request to http://localhost:8000/chat/completions (attempt 1/3)
Sending request to http://localhost:8000/chat/completions (attempt 1/3)
Sending request to http://localhost:8000/chat/completions (attempt 1/3)
Sending request to http://localhost:8000/chat/completions (attempt 1/3)
Sending request to http://localhost:8000/chat/completions (attempt 1/3)
Sending request to http://localhost:8000/chat/completions (attem

...

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Text Informativeness F1: 0.8716
Text Category F1: 0.6423
Image Informativeness F1: 0.9201
Image Category F1: 0.6860
Damage Severity F1: 0.4011

--- 3_9_2017 (439 samples) ---
Text Informativeness F1: 0.8827
Text Category F1: 0.5784
Image Informativeness F1: 0.9002
Image Category F1: 0.6805
Damage Severity F1: 0.1864

--- 4_9_2017 (392 samples) ---
Text Informativeness F1: 0.8664
Text Category F1: 0.6145
Image Informativeness F1: 0.8778
Image Category F1: 0.7192
Damage Severity F1: 0.2834

--- 5_9_2017 (395 samples) ---
Text Informativeness F1: 0.8519
Text Category F1: 0.5997
Image Informativeness F1: 0.8913
Image Category F1: 0.7224
Damage Severity F1: 0.2938

--- 6_9_2017 (359 samples) ---
Text Informativeness F1: 0.8112
Text Category F1: 0.6107
Image Informativeness F1: 0.8325
Image Category F1: 0.7077
Damage Severity F1: 0.3602

--- 8_9_2017 (253 samples) ---
Text Informativeness F1: 0.8222
Text Category F1: 0.6014
Image Informativeness F1: 0.8692
Image Category F1: 0.7894
Damage Se

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated. If you don't see the download prompt, the file is available at:
  /content/disaster_analysis_results.zip

Key result files:
  - all_results.csv: files.download('/content/llama.cpp/results/all_results.csv')
  - all_results_metrics.json: files.download('/content/llama.cpp/results/all_results_metrics.json')
  - all_results_metrics_by_disaster.csv: files.download('/content/llama.cpp/results/all_results_metrics_by_disaster.csv')
  - all_results_confusion_matrices.png: files.download('/content/llama.cpp/results/all_results_confusion_matrices.png')
  - all_results_disaster_comparison.png: files.download('/content/llama.cpp/results/all_results_disaster_comparison.png')


In [ ]:
# Run this cell to evaluate and download your existing results

import os
import glob
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
from google.colab import files
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import numpy as np

# Configure paths
OUTPUT_DIR = "/content/llama.cpp/results/"
results_file = os.path.join(OUTPUT_DIR, "all_results.csv")

# Create backup function
def create_backup(output_dir, backup_name=None):
    """Create a zip file of results without downloading it"""
    if backup_name is None:
        # Create a timestamp for the backup
        import time
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        backup_name = f"disaster_results_{timestamp}.zip"

    # Path for the zip file
    zip_path = f"/content/{backup_name}"

    print(f"\nCreating backup: {backup_name}...")

    # Create the zip file
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Walk through the results directory
        for root, _, files_list in os.walk(output_dir):
            for file in files_list:
                # Get the full path of the file
                file_path = os.path.join(root, file)
                # Calculate path inside the zip file
                arcname = os.path.relpath(file_path, os.path.dirname(output_dir))
                # Add file to the zip
                zipf.write(file_path, arcname)

    print(f"Backup created: {zip_path}")
    print(f"To download, run: files.download('{zip_path}')")

    return zip_path

# Helper function to calculate F1 scores
def calculate_f1_scores(results_df):
    """Calculate F1 scores comparing model predictions to ground truth"""
    metrics = {}

    # Filter out rows with missing predictions or ground truth
    has_text_preds = pd.notna(results_df['predicted_text_info']) & pd.notna(results_df['predicted_text_human'])
    has_text_truth = pd.notna(results_df['true_text_info']) & pd.notna(results_df['true_text_human'])
    has_image_preds = pd.notna(results_df['predicted_image_info']) & pd.notna(results_df['predicted_image_human'])
    has_image_truth = pd.notna(results_df['true_image_info']) & pd.notna(results_df['true_image_human'])

    # Create filtered dataframes with only valid rows
    text_df = results_df[has_text_preds & has_text_truth]
    image_df = results_df[has_image_preds & has_image_truth]

    # Calculate text informativeness F1 (binary classification)
    if len(text_df) > 0:
        text_info_f1 = f1_score(
            text_df['true_text_info'],
            text_df['predicted_text_info'],
            pos_label='informative',
            average='binary',
            zero_division=0
        )
        metrics['text_info_f1'] = text_info_f1

        # Calculate text category F1 (multiclass classification)
        text_human_f1 = f1_score(
            text_df['true_text_human'],
            text_df['predicted_text_human'],
            average='weighted',
            zero_division=0
        )
        metrics['text_human_f1'] = text_human_f1

    # Calculate image informativeness F1 (binary classification)
    if len(image_df) > 0:
        image_info_f1 = f1_score(
            image_df['true_image_info'],
            image_df['predicted_image_info'],
            pos_label='informative',
            average='binary',
            zero_division=0
        )
        metrics['image_info_f1'] = image_info_f1

        # Calculate image category F1 (multiclass classification)
        image_human_f1 = f1_score(
            image_df['true_image_human'],
            image_df['predicted_image_human'],
            average='weighted',
            zero_division=0
        )
        metrics['image_human_f1'] = image_human_f1

    # Calculate damage severity F1 (only for rows with damage annotations)
    if 'true_image_damage' in results_df.columns:
        has_damage_truth = pd.notna(results_df['true_image_damage'])
        has_damage_pred = pd.notna(results_df['predicted_image_damage'])
        damage_df = results_df[has_damage_truth & has_damage_pred]

        if len(damage_df) > 0:
            damage_f1 = f1_score(
                damage_df['true_image_damage'],
                damage_df['predicted_image_damage'],
                average='weighted',
                zero_division=0
            )
            metrics['damage_f1'] = damage_f1

    return metrics

# Function to generate confusion matrices
def generate_confusion_matrices(df, output_file):
    """Generate and save confusion matrices for model predictions"""
    # Create a figure with multiple subplots
    plt.figure(figsize=(20, 15))

    # Filter valid rows
    has_text_preds = pd.notna(df['predicted_text_info']) & pd.notna(df['predicted_text_human'])
    has_text_truth = pd.notna(df['true_text_info']) & pd.notna(df['true_text_human'])
    has_image_preds = pd.notna(df['predicted_image_info']) & pd.notna(df['predicted_image_human'])
    has_image_truth = pd.notna(df['true_image_info']) & pd.notna(df['true_image_human'])

    text_df = df[has_text_preds & has_text_truth]
    image_df = df[has_image_preds & has_image_truth]

    # Plot Text Informativeness Confusion Matrix
    if len(text_df) > 0:
        plt.subplot(2, 3, 1)
        conf_mat = confusion_matrix(
            text_df['true_text_info'],
            text_df['predicted_text_info'],
            normalize='true'  # Normalize by row (ground truth)
        )
        sns.heatmap(conf_mat, annot=True, fmt='.2f', cmap='Blues',
                   xticklabels=['not_informative', 'informative'],
                   yticklabels=['not_informative', 'informative'])
        plt.title('Text Informativeness')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')

    # Plot Image Informativeness Confusion Matrix
    if len(image_df) > 0:
        plt.subplot(2, 3, 2)
        conf_mat = confusion_matrix(
            image_df['true_image_info'],
            image_df['predicted_image_info'],
            normalize='true'  # Normalize by row (ground truth)
        )
        sns.heatmap(conf_mat, annot=True, fmt='.2f', cmap='Blues',
                   xticklabels=['not_informative', 'informative'],
                   yticklabels=['not_informative', 'informative'])
        plt.title('Image Informativeness')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')

    # Plot Text Category Confusion Matrix (top categories)
    if len(text_df) > 0:
        plt.subplot(2, 3, 3)
        # Get the most common categories (top 5)
        top_cats = pd.concat([text_df['true_text_human'], text_df['predicted_text_human']]).value_counts().head(5).index
        # Filter to only include top categories
        filtered_text_df = text_df[text_df['true_text_human'].isin(top_cats) & text_df['predicted_text_human'].isin(top_cats)]

        if len(filtered_text_df) > 0:
            conf_mat = confusion_matrix(
                filtered_text_df['true_text_human'],
                filtered_text_df['predicted_text_human'],
                normalize='true',  # Normalize by row (ground truth)
                labels=top_cats
            )
            sns.heatmap(conf_mat, annot=True, fmt='.2f', cmap='Blues',
                      xticklabels=top_cats, yticklabels=top_cats)
            plt.title('Text Category (Top 5)')
            plt.ylabel('True Label')
            plt.xlabel('Predicted Label')
            plt.xticks(rotation=45)
            plt.yticks(rotation=45)

    # Plot Image Category Confusion Matrix (top categories)
    if len(image_df) > 0:
        plt.subplot(2, 3, 4)
        # Get the most common categories (top 5)
        top_cats = pd.concat([image_df['true_image_human'], image_df['predicted_image_human']]).value_counts().head(5).index
        # Filter to only include top categories
        filtered_image_df = image_df[image_df['true_image_human'].isin(top_cats) & image_df['predicted_image_human'].isin(top_cats)]

        if len(filtered_image_df) > 0:
            conf_mat = confusion_matrix(
                filtered_image_df['true_image_human'],
                filtered_image_df['predicted_image_human'],
                normalize='true',  # Normalize by row (ground truth)
                labels=top_cats
            )
            sns.heatmap(conf_mat, annot=True, fmt='.2f', cmap='Blues',
                      xticklabels=top_cats, yticklabels=top_cats)
            plt.title('Image Category (Top 5)')
            plt.ylabel('True Label')
            plt.xlabel('Predicted Label')
            plt.xticks(rotation=45)
            plt.yticks(rotation=45)

    # Plot Damage Severity Confusion Matrix
    if 'true_image_damage' in df.columns:
        has_damage_truth = pd.notna(df['true_image_damage'])
        has_damage_pred = pd.notna(df['predicted_image_damage'])
        damage_df = df[has_damage_truth & has_damage_pred]

        if len(damage_df) > 0:
            plt.subplot(2, 3, 5)
            # Get common damage labels
            damage_labels = sorted(list(set(damage_df['true_image_damage'].unique()) | set(damage_df['predicted_image_damage'].unique())))

            conf_mat = confusion_matrix(
                damage_df['true_image_damage'],
                damage_df['predicted_image_damage'],
                normalize='true',  # Normalize by row (ground truth)
                labels=damage_labels
            )
            sns.heatmap(conf_mat, annot=True, fmt='.2f', cmap='Blues',
                      xticklabels=damage_labels, yticklabels=damage_labels)
            plt.title('Damage Severity')
            plt.ylabel('True Label')
            plt.xlabel('Predicted Label')
            plt.xticks(rotation=45)
            plt.yticks(rotation=45)

    # Add overall title
    plt.suptitle('Confusion Matrices - Model vs. Ground Truth', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to make room for the suptitle

    # Save to file
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.close()

# Function to evaluate results
def evaluate_results(results_file):
    """Evaluate model performance against ground truth with detailed metrics"""
    if not os.path.exists(results_file):
        print(f"Results file not found: {results_file}")
        return

    df = pd.read_csv(results_file)
    print(f"Loaded {len(df)} results from {results_file}")

    # Basic statistics

    # Count by disaster type
    if 'disaster_type' in df.columns:
        disaster_counts = df['disaster_type'].value_counts()
        print("\nResults by disaster type:")
        for disaster, count in disaster_counts.items():
            print(f"  - {disaster}: {count}")

    # Count predictions by category
    print("\nText category predictions:")
    if 'predicted_text_human' in df.columns:
        text_counts = df['predicted_text_human'].value_counts()
        for category, count in text_counts.items():
            print(f"  - {category}: {count} ({count/len(df):.1%})")

    print("\nImage category predictions:")
    if 'predicted_image_human' in df.columns:
        image_counts = df['predicted_image_human'].value_counts()
        for category, count in image_counts.items():
            print(f"  - {category}: {count} ({count/len(df):.1%})")

    print("\nDamage severity predictions:")
    if 'predicted_image_damage' in df.columns:
        damage_counts = df['predicted_image_damage'].value_counts()
        for severity, count in damage_counts.items():
            print(f"  - {severity}: {count} ({count/len(df):.1%})")

    # Performance evaluation with F1 scores
    print("\n===== MODEL PERFORMANCE (F1 SCORES) =====")
    metrics = calculate_f1_scores(df)

    if 'text_info_f1' in metrics:
        print(f"Text Informativeness F1: {metrics['text_info_f1']:.4f}")
    if 'text_human_f1' in metrics:
        print(f"Text Category F1: {metrics['text_human_f1']:.4f}")
    if 'image_info_f1' in metrics:
        print(f"Image Informativeness F1: {metrics['image_info_f1']:.4f}")
    if 'image_human_f1' in metrics:
        print(f"Image Category F1: {metrics['image_human_f1']:.4f}")
    if 'damage_f1' in metrics:
        print(f"Damage Severity F1: {metrics['damage_f1']:.4f}")

    # Save metrics to file
    metrics_file = results_file.replace('.csv', '_metrics.json')
    with open(metrics_file, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"Metrics saved to {metrics_file}")

    # Generate detailed classification reports
    print("\n===== DETAILED CLASSIFICATION REPORTS =====")

    # Filter valid rows
    has_text_preds = pd.notna(df['predicted_text_info']) & pd.notna(df['predicted_text_human'])
    has_text_truth = pd.notna(df['true_text_info']) & pd.notna(df['true_text_human'])
    has_image_preds = pd.notna(df['predicted_image_info']) & pd.notna(df['predicted_image_human'])
    has_image_truth = pd.notna(df['true_image_info']) & pd.notna(df['true_image_human'])

    text_df = df[has_text_preds & has_text_truth]
    image_df = df[has_image_preds & has_image_truth]

    if len(text_df) > 0:
        print("\nText Informativeness Classification Report:")
        print(classification_report(
            text_df['true_text_info'],
            text_df['predicted_text_info'],
            zero_division=0
        ))

        print("\nText Category Classification Report:")
        print(classification_report(
            text_df['true_text_human'],
            text_df['predicted_text_human'],
            zero_division=0
        ))

    if len(image_df) > 0:
        print("\nImage Informativeness Classification Report:")
        print(classification_report(
            image_df['true_image_info'],
            image_df['predicted_image_info'],
            zero_division=0
        ))

        print("\nImage Category Classification Report:")
        print(classification_report(
            image_df['true_image_human'],
            image_df['predicted_image_human'],
            zero_division=0
        ))

    if 'true_image_damage' in df.columns:
        has_damage_truth = pd.notna(df['true_image_damage'])
        has_damage_pred = pd.notna(df['predicted_image_damage'])
        damage_df = df[has_damage_truth & has_damage_pred]

        if len(damage_df) > 0:
            print("\nDamage Severity Classification Report:")
            print(classification_report(
                damage_df['true_image_damage'],
                damage_df['predicted_image_damage'],
                zero_division=0
            ))

    # Create and save confusion matrices
    try:
        generate_confusion_matrices(df, results_file.replace('.csv', '_confusion_matrices.png'))
        print(f"Confusion matrices saved to {results_file.replace('.csv', '_confusion_matrices.png')}")
    except Exception as e:
        print(f"Error generating confusion matrices: {e}")

    return metrics

# Function to evaluate by disaster type
def evaluate_by_disaster_type(results_file):
    """Calculate and compare F1 scores across different disaster types"""
    if not os.path.exists(results_file):
        print(f"Results file not found: {results_file}")
        return

    df = pd.read_csv(results_file)

    if 'disaster_type' not in df.columns:
        print("Disaster type information not available in results")
        return

    # Get unique disaster types
    disaster_types = df['disaster_type'].unique()

    print(f"\n===== PERFORMANCE BY DISASTER TYPE ({len(disaster_types)} types) =====")

    # Initialize results dictionary
    disaster_metrics = {}

    for disaster in disaster_types:
        # Filter data for this disaster type
        disaster_df = df[df['disaster_type'] == disaster]

        print(f"\n--- {disaster} ({len(disaster_df)} samples) ---")

        # Calculate metrics for this disaster type
        metrics = calculate_f1_scores(disaster_df)
        disaster_metrics[disaster] = metrics

        # Print metrics
        if 'text_info_f1' in metrics:
            print(f"Text Informativeness F1: {metrics['text_info_f1']:.4f}")
        if 'text_human_f1' in metrics:
            print(f"Text Category F1: {metrics['text_human_f1']:.4f}")
        if 'image_info_f1' in metrics:
            print(f"Image Informativeness F1: {metrics['image_info_f1']:.4f}")
        if 'image_human_f1' in metrics:
            print(f"Image Category F1: {metrics['image_human_f1']:.4f}")
        if 'damage_f1' in metrics:
            print(f"Damage Severity F1: {metrics['damage_f1']:.4f}")

    # Save metrics by disaster type
    metrics_by_disaster_file = results_file.replace('.csv', '_metrics_by_disaster.json')
    with open(metrics_by_disaster_file, 'w') as f:
        json.dump(disaster_metrics, f, indent=2)
    print(f"\nMetrics by disaster type saved to {metrics_by_disaster_file}")

    # Create a comparison dataframe for visualization
    try:
        # Convert to DataFrame for easier comparison
        metrics_df = pd.DataFrame.from_dict(disaster_metrics, orient='index')

        # Save as CSV
        metrics_csv = results_file.replace('.csv', '_metrics_by_disaster.csv')
        metrics_df.to_csv(metrics_csv)
        print(f"Metrics comparison CSV saved to {metrics_csv}")

        # Create a heatmap visualization
        plt.figure(figsize=(12, 8))
        sns.heatmap(metrics_df, annot=True, fmt='.3f', cmap='viridis', linewidths=.5)
        plt.title('F1 Scores by Disaster Type')
        plt.tight_layout()

        # Save the heatmap
        heatmap_file = results_file.replace('.csv', '_disaster_comparison.png')
        plt.savefig(heatmap_file, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Comparison heatmap saved to {heatmap_file}")
    except Exception as e:
        print(f"Error creating comparison visualization: {e}")

    return disaster_metrics

# Main execution
print("\n===== EVALUATING RESULTS =====")

# Check if results file exists
if os.path.exists(results_file):
    print(f"Found results file: {results_file}")

    # Overall evaluation - this gives metrics across all disaster types combined
    print("\n===== OVERALL PERFORMANCE (ALL DISASTERS COMBINED) =====")
    overall_metrics = evaluate_results(results_file)

    # Per-disaster type evaluation
    print("\n===== PERFORMANCE BY DISASTER TYPE =====")
    disaster_metrics = evaluate_by_disaster_type(results_file)

    # Create and download a zip with all result files
    print("\nCreating results package for download...")
    results_backup = create_backup(OUTPUT_DIR, "disaster_analysis_results.zip")

    # Provide download command
    print(f"\nTo download all results, run: files.download('{results_backup}')")

    # Also provide individual download commands for key files
    key_files = [
        results_file,
        results_file.replace('.csv', '_metrics.json'),
        results_file.replace('.csv', '_metrics_by_disaster.csv'),
        results_file.replace('.csv', '_confusion_matrices.png'),
        results_file.replace('.csv', '_disaster_comparison.png')
    ]

    print("\nKey result files:")
    for f in key_files:
        if os.path.exists(f):
            filename = os.path.basename(f)
            print(f"  - {filename}: files.download('{f}')")

    # Print a summary of the most important metrics for quick reference
    print("\n===== SUMMARY OF KEY METRICS =====")
    print("Overall Performance (F1 Scores):")

    if overall_metrics and isinstance(overall_metrics, dict):
        for metric_name, value in overall_metrics.items():
            if isinstance(value, (int, float)):
                print(f"  - {metric_name}: {value:.4f}")
    else:
        print("  No overall metrics available")

    # Extract best and worst performing disaster types
    if disaster_metrics and isinstance(disaster_metrics, dict) and len(disaster_metrics) > 0:
        print("\nTop Performing Disaster Types:")

        # Find disaster types with metrics
        disasters_with_metrics = {}
        for disaster, metrics in disaster_metrics.items():
            if metrics and isinstance(metrics, dict) and 'image_info_f1' in metrics:
                disasters_with_metrics[disaster] = metrics.get('image_info_f1', 0)

        # Sort and print top 3
        if disasters_with_metrics:
            top_disasters = sorted(disasters_with_metrics.items(), key=lambda x: x[1], reverse=True)[:3]
            for disaster, score in top_disasters:
                print(f"  - {disaster}: {score:.4f} (Image Informativeness F1)")
        else:
            print("  No disaster-specific metrics available")

else:
    print(f"Results file not found: {results_file}")

    # Look for any results files
    csv_files = glob.glob(os.path.join(OUTPUT_DIR, "*.csv"))
    if csv_files:
        print("\nFound these result files:")
        for csv_file in csv_files:
            print(f"  - {os.path.basename(csv_file)}")

        # Suggest specific files to evaluate
        if len(csv_files) > 0:
            print("\nTo evaluate a specific file, run:")
            print(f"evaluate_results('{csv_files[0]}')")
    else:
        print("\nNo result files found in the output directory.")


===== EVALUATING RESULTS =====
Found results file: /content/llama.cpp/results/all_results.csv

===== OVERALL PERFORMANCE (ALL DISASTERS COMBINED) =====
Loaded 18082 results from /content/llama.cpp/results/all_results.csv

Results by disaster type:
  - 18_9_2017: 1692
  - 19_9_2017: 1533
  - 17_9_2017: 927
  - 7_9_2017: 882
  - 20_9_2017: 694
  - 3_9_2017: 439
  - 25_9_2017: 420
  - 23_9_2017: 411
  - 21_9_2017: 396
  - 5_9_2017: 395
  - 4_9_2017: 392
  - 24_9_2017: 382
  - 27_8_2017: 369
  - 6_9_2017: 359
  - 20_10_2017: 340
  - 13_11_2017: 323
  - 10_9_2017: 299
  - 21_10_2017: 258
  - 11_9_2017: 257
  - 27_9_2017: 254
  - 8_9_2017: 253
  - 23_10_2017: 242
  - 26_10_2017: 241
  - 25_10_2017: 225
  - 11_10_2017: 221
  - 12_9_2017: 220
  - 10_10_2017: 214
  - 14_9_2017: 212
  - 24_10_2017: 211
  - 28_9_2017: 210
  - 16_10_2017: 207
  - 13_9_2017: 202
  - 6_10_2017: 198
  - 9_9_2017: 197
  - 15_9_2017: 195
  - 5_10_2017: 192
  - 29_9_2017: 174
  - 30_9_2017: 172
  - 22_10_2017: 170
  - 

In [ ]:
# Run this cell to calculate F1 scores comparing your model predictions to ground truth

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import os
import zipfile
from sklearn.metrics import f1_score

# Path to your results
results_file = "/content/llama.cpp/results/all_results.csv"

# Function to calculate F1 scores between model predictions and ground truth
def calculate_f1_scores(df):
    """Calculate F1 scores comparing model predictions to ground truth annotations"""
    results = {}

    # Text informativeness F1
    if 'predicted_text_info' in df.columns and 'true_text_info' in df.columns:
        valid_rows = df.dropna(subset=['predicted_text_info', 'true_text_info'])
        if len(valid_rows) > 0:
            text_info_f1 = f1_score(
                valid_rows['true_text_info'],
                valid_rows['predicted_text_info'],
                pos_label='informative',
                average='binary',
                zero_division=0
            )
            results['Text Informativeness F1'] = text_info_f1

            # Also calculate simple agreement
            agreement = (valid_rows['true_text_info'] == valid_rows['predicted_text_info']).mean() * 100
            results['Text Informativeness Agreement (%)'] = agreement

    # Text category F1
    if 'predicted_text_human' in df.columns and 'true_text_human' in df.columns:
        valid_rows = df.dropna(subset=['predicted_text_human', 'true_text_human'])
        if len(valid_rows) > 0:
            text_cat_f1 = f1_score(
                valid_rows['true_text_human'],
                valid_rows['predicted_text_human'],
                average='weighted',
                zero_division=0
            )
            results['Text Category F1'] = text_cat_f1

            # Also calculate simple agreement
            agreement = (valid_rows['true_text_human'] == valid_rows['predicted_text_human']).mean() * 100
            results['Text Category Agreement (%)'] = agreement

    # Image informativeness F1
    if 'predicted_image_info' in df.columns and 'true_image_info' in df.columns:
        valid_rows = df.dropna(subset=['predicted_image_info', 'true_image_info'])
        if len(valid_rows) > 0:
            image_info_f1 = f1_score(
                valid_rows['true_image_info'],
                valid_rows['predicted_image_info'],
                pos_label='informative',
                average='binary',
                zero_division=0
            )
            results['Image Informativeness F1'] = image_info_f1

            # Also calculate simple agreement
            agreement = (valid_rows['true_image_info'] == valid_rows['predicted_image_info']).mean() * 100
            results['Image Informativeness Agreement (%)'] = agreement

    # Image category F1
    if 'predicted_image_human' in df.columns and 'true_image_human' in df.columns:
        valid_rows = df.dropna(subset=['predicted_image_human', 'true_image_human'])
        if len(valid_rows) > 0:
            image_cat_f1 = f1_score(
                valid_rows['true_image_human'],
                valid_rows['predicted_image_human'],
                average='weighted',
                zero_division=0
            )
            results['Image Category F1'] = image_cat_f1

            # Also calculate simple agreement
            agreement = (valid_rows['true_image_human'] == valid_rows['predicted_image_human']).mean() * 100
            results['Image Category Agreement (%)'] = agreement

    # Damage severity F1
    if 'predicted_image_damage' in df.columns and 'true_image_damage' in df.columns:
        valid_rows = df.dropna(subset=['predicted_image_damage', 'true_image_damage'])
        if len(valid_rows) > 0:
            damage_f1 = f1_score(
                valid_rows['true_image_damage'],
                valid_rows['predicted_image_damage'],
                average='weighted',
                zero_division=0
            )
            results['Damage Severity F1'] = damage_f1

            # Also calculate simple agreement
            agreement = (valid_rows['true_image_damage'] == valid_rows['predicted_image_damage']).mean() * 100
            results['Damage Severity Agreement (%)'] = agreement

    return results

# Load the results
if os.path.exists(results_file):
    df = pd.read_csv(results_file)
    print(f"Loaded {len(df)} results from {results_file}")
else:
    print(f"Results file not found: {results_file}")

    # Try to find any results
    import glob
    result_files = glob.glob("/content/llama.cpp/results/*.csv")
    if result_files:
        print("Found these other result files:")
        for f in result_files:
            print(f"  - {os.path.basename(f)}")

        # Use the first one we find
        results_file = result_files[0]
        df = pd.read_csv(results_file)
        print(f"Using {os.path.basename(results_file)} with {len(df)} results")
    else:
        print("No results found. Please check the file path.")
        df = None

if df is not None:
    # Calculate F1 scores comparing model predictions to ground truth
    print("\n===== MODEL vs GROUND TRUTH F1 SCORES =====")
    metrics = calculate_f1_scores(df)

    # Print all metrics
    for metric, value in metrics.items():
        if 'Agreement' in metric:
            print(f"{metric}: {value:.1f}%")
        else:
            print(f"{metric}: {value:.4f}")

    # Create confusion matrices to visualize model vs ground truth
    print("\n===== MODEL vs GROUND TRUTH CONFUSION MATRICES =====")

    # Function to create and save confusion matrix
    def create_confusion_matrix(df, pred_col, true_col, title):
        valid_rows = df.dropna(subset=[pred_col, true_col])
        if len(valid_rows) == 0:
            print(f"No valid data for {title}")
            return None

        # Get all unique categories
        categories = sorted(list(set(valid_rows[pred_col].unique()) | set(valid_rows[true_col].unique())))

        # Create confusion matrix
        cm = pd.crosstab(
            valid_rows[true_col],
            valid_rows[pred_col],
            normalize='index',
            rownames=['Ground Truth'],
            colnames=['Model Prediction']
        )

        # Ensure all categories are present
        for cat in categories:
            if cat not in cm.index:
                cm.loc[cat] = 0
            if cat not in cm.columns:
                cm[cat] = 0

        # Sort to ensure consistent order
        cm = cm.loc[categories, categories]

        # Plot
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='.1%', cmap='Blues', vmin=0, vmax=1)
        plt.title(f"{title}: Model vs Ground Truth")
        plt.tight_layout()

        # Save
        filename = f"{title.lower().replace(' ', '_')}_confusion.png"
        plt.savefig(filename)
        plt.close()

        print(f"Saved {title} confusion matrix to {filename}")
        return filename

    # Create confusion matrices for all classification tasks
    matrix_files = []

    # Text informativeness
    if 'predicted_text_info' in df.columns and 'true_text_info' in df.columns:
        filename = create_confusion_matrix(df, 'predicted_text_info', 'true_text_info', 'Text_Informativeness')
        if filename:
            matrix_files.append(filename)

    # Text category
    if 'predicted_text_human' in df.columns and 'true_text_human' in df.columns:
        filename = create_confusion_matrix(df, 'predicted_text_human', 'true_text_human', 'Text_Category')
        if filename:
            matrix_files.append(filename)

    # Image informativeness
    if 'predicted_image_info' in df.columns and 'true_image_info' in df.columns:
        filename = create_confusion_matrix(df, 'predicted_image_info', 'true_image_info', 'Image_Informativeness')
        if filename:
            matrix_files.append(filename)

    # Image category
    if 'predicted_image_human' in df.columns and 'true_image_human' in df.columns:
        filename = create_confusion_matrix(df, 'predicted_image_human', 'true_image_human', 'Image_Category')
        if filename:
            matrix_files.append(filename)

    # Damage severity
    if 'predicted_image_damage' in df.columns and 'true_image_damage' in df.columns:
        filename = create_confusion_matrix(df, 'predicted_image_damage', 'true_image_damage', 'Damage_Severity')
        if filename:
            matrix_files.append(filename)

    # Zip all matrices for easy download
    if matrix_files:
        zip_path = "/content/model_vs_ground_truth_matrices.zip"
        with zipfile.ZipFile(zip_path, 'w') as zipf:
            for file in matrix_files:
                zipf.write(file)

        print(f"\nAll matrices saved to {zip_path}")
        print(f"To download: files.download('{zip_path}')")

    # Save metrics to CSV
    metrics_df = pd.DataFrame(metrics.items(), columns=['Metric', 'Value'])
    metrics_df.to_csv('model_ground_truth_metrics.csv', index=False)
    print("Metrics saved to model_ground_truth_metrics.csv")
    print("To download: files.download('model_ground_truth_metrics.csv')")

Loaded 18082 results from /content/llama.cpp/results/all_results.csv

===== MODEL vs GROUND TRUTH F1 SCORES =====
Text Informativeness F1: 0.8415
Text Informativeness Agreement (%): 78.3%
Text Category F1: 0.6106
Text Category Agreement (%): 61.0%
Image Informativeness F1: 0.8417
Image Informativeness Agreement (%): 83.1%
Image Category F1: 0.7213
Image Category Agreement (%): 71.4%
Damage Severity F1: 0.4372
Damage Severity Agreement (%): 37.9%

===== MODEL vs GROUND TRUTH CONFUSION MATRICES =====
Saved Text_Informativeness confusion matrix to text_informativeness_confusion.png
Saved Text_Category confusion matrix to text_category_confusion.png
Saved Image_Informativeness confusion matrix to image_informativeness_confusion.png
Saved Image_Category confusion matrix to image_category_confusion.png
Saved Damage_Severity confusion matrix to damage_severity_confusion.png

All matrices saved to /content/model_vs_ground_truth_matrices.zip
To download: files.download('/content/model_vs_ground

In [ ]:
files.download('/content/model_vs_ground_truth_matrices.zip')
files.download('/content/disaster_analysis_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>